# Unified training notebook — dpo-safety-representations

One notebook for every training stage (`M0` needs no training), config-driven — orchestrates
repository code (`src/training/train_sft.py`, `src/training/train_dpo.py`), no training logic
lives in this notebook itself. Stage list, dependency order, config paths, and data-prep
requirements all come from `src/training/stage_registry.py` (single source of truth, also used
by `src/reproduce.py`) — editing that file is the only place a new stage needs to be added.

**Supported stages:** `M1`, `M2`, `M3`, `M3_direct`, `M1_alt`, `M2_alt`, `M3_alt`, `M3_direct_alt`.

Set `STAGES_TO_RUN` and `DRY_RUN` in the Configuration cell below, then run top to bottom.
**If Colab disconnects mid-training:** reconnect, rerun this whole notebook top to bottom with the
same `STAGES_TO_RUN` — every stage resumes automatically from its own last checkpoint in Drive
(and any stage that already finished is skipped, not retrained).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = "https://github.com/urosavurdic/dpo-safety-representations.git"
REPO_DIR = "/content/dpo-safety-representations"
BRANCH = "main"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -y torchao

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
!pytest tests/ -v

In [ ]:
import wandb
wandb.login()

In [ ]:
from huggingface_hub import login
login()

## Configuration

Edit `STAGES_TO_RUN` and `DRY_RUN`, then run this cell and everything below it. You only need to
list the stage(s) you actually want the *final* model for — prerequisites (e.g. `M3` needs `M1`
then `M2` trained and pushed first) are pulled in and ordered automatically.

In [ ]:
from src.training.stage_registry import TRAINING_STAGES, resolve_run_order, data_prep_commands_for

# ---- EDIT THIS ----
STAGES_TO_RUN = ["M3"]   # e.g. ["M3", "M3_direct"] -- see printed "Valid stages" below for the full list
DRY_RUN = True            # True = tiny smoke-test config (few steps, no push). Flip to False for the real run.
# --------------------

print("Valid stages:", sorted(TRAINING_STAGES.keys()))
RUN_ORDER = resolve_run_order(STAGES_TO_RUN)
print(f"\nRequested: {STAGES_TO_RUN}")
print(f"Resolved run order (prerequisites included): {RUN_ORDER}")
print(f"Mode: {'DRY RUN (smoke test)' if DRY_RUN else 'REAL TRAINING'}")

## Data prep

Builds only what `RUN_ORDER` actually needs, skips anything already present (most of these are
already committed to the repo — only `M1_alt`'s Dolly-derived data is new and will actually build
here the first time).

In [ ]:
import os

for description, output_path, command in data_prep_commands_for(RUN_ORDER):
    if os.path.exists(output_path):
        print(f"[skip] {description}: {output_path} already present")
    else:
        print(f"[build] {description} -> {output_path}")
        !{command}

## Train

Runs each stage in `RUN_ORDER`, in order — dry-run config if `DRY_RUN`, real config otherwise.
Skips any stage whose `final/` adapter directory already exists (safe to rerun this cell after a
disconnect, or after adding a new stage to `STAGES_TO_RUN` that shares prerequisites with one
you've already trained).

In [ ]:
import yaml

for stage in RUN_ORDER:
    spec = TRAINING_STAGES[stage]
    config_path = spec["dryrun_config"] if DRY_RUN else spec["config"]
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    final_dir = os.path.join(cfg["output"]["base_dir"], "final")

    if os.path.exists(final_dir) and os.listdir(final_dir):
        print(f"\n=== {stage}: final adapter already present at {final_dir}, skipping ===")
        continue

    module = "src.training.train_sft" if spec["kind"] == "sft" else "src.training.train_dpo"
    print(f"\n=== {stage} ({spec['kind']}, {'dry run' if DRY_RUN else 'real'}): python -m {module} --config {config_path} ===")
    !python -m {module} --config {config_path}

## Verify artifacts

Confirms every requested stage's final adapter actually landed.

In [ ]:
for stage in RUN_ORDER:
    spec = TRAINING_STAGES[stage]
    config_path = spec["dryrun_config"] if DRY_RUN else spec["config"]
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    final_dir = os.path.join(cfg["output"]["base_dir"], "final")
    status = "OK" if os.path.exists(final_dir) and os.listdir(final_dir) else "MISSING"
    print(f"{stage:16s} [{status}]  {final_dir}")
    if status == "OK":
        print(f"                 {os.listdir(final_dir)}")